# How to run cromwell WDL files in the VWB

This notebook is to show how to run WDL job using Cromwell run mode. 

Please go to the workspace lobby -> `Workflows` -> to run this WDL job using server mode.

# 1st step is to setup the variables

Assuming you have run any of these notebooks

- /home/jupyter/workspace/aou-tutorial-notebooks/I. Getting Started with Verily Workbench/01_1. Getting Started with Workbench (Python).ipynb
- /home/jupyter/workspace/aou-tutorial-notebooks/Setting_Env_Variables_01_readme_python/R

So you have these two files in this directory. If not, please go through those notebooks mentioned above to copy the files over

And then you can run the cells below to load the variables.

In [ ]:
# %run /home/jupyter/workspace/aou-tutorial-notebooks/Setting_Env_Variables.ipynb

In [ ]:
%run Setting_Env_Variables_p2.py


In [ ]:
!echo $PWD

In [ ]:
!echo ${WORKSPACE_BUCKET}

# check the default cromwell jar files

In [ ]:
!java --version

In [ ]:
# check the default JAR file
!echo $CROMWELL_JAR

Set up this variable `$WOMTOOL_JAR` as well to use it later

In [ ]:
%env WOMTOOL_JAR=/opt/workbench-tools/binaries/share/cromwell/womtool.jar

In [ ]:
!ls /home/dataproc/.local/share/cromwell/

In [ ]:
!echo $WOMTOOL_JAR

In [ ]:
!java -jar $WOMTOOL_JAR --version

# Second step is to write the config file 
where three variables including `${GOOGLE_PROJECT}`, `${WORKSPACE_BUCKET}` and `${PET_SA_EMAIL}` are needed

In [ ]:
%%bash
cat > cromwell.conf <<EOF
backend {
  default = "GCPBATCH"
  providers {

    # Disables the Local backend
    Local.config.root = "/dev/null"

    GCPBATCH {
     
        actor-factory = "cromwell.backend.google.batch.GcpBatchBackendLifecycleActorFactory"

      config {
        project = "${GOOGLE_PROJECT}"
        concurrent-job-limit = 10
        root = "${WORKSPACE_BUCKET}/workflows/cromwell-executions"

                        
          virtual-private-cloud {
          network-name = "projects/${GOOGLE_PROJECT}/global/networks/network"
          subnetwork-name = "projects/${GOOGLE_PROJECT}/regions/*/subnetworks/subnetwork"
                   
        }

        batch {
          auth = "application-default"
          compute-service-account = "${PET_SA_EMAIL}"
          location = "us-central1"
        }

        default-runtime-attributes {
          noAddress: true
        }
        
        filesystems {
          gcs {
              auth = "application-default"
          }
        }
      }
    }
  }
}
EOF

In [ ]:
!ls /home/jupyter/cromwell.conf

Double check the config file

In [ ]:
!cat ~/cromwell.conf

# Third step to write a wdl/json file

In [ ]:
%%writefile make_lmna_vat_table.wdl
version 1.0

# Example workflow

workflow hello_world {
    call lmna_vat
}

task lmna_vat {
    command {
        python make_lmna_vat_table.py

    }
    output {
        File output_greeting = stdout()
    }
    runtime {
    docker: "ubuntu:latest"
    memory: "8 GB"
    disks: "local-disk 200 HDD"
    bootDiskSizeGb: 50
    }
}


In [ ]:
%%writefile make_lmna_vat_table.json
{}

In [ ]:
!ls hello_world*.*

Set up variables for the WDL/json files

In [ ]:
%env wdl_filename=hello_world.wdl

In [ ]:
%env json_filename = hello_world.json

In [ ]:
!echo $wdl_filename

In [ ]:
!echo $json_filename

# Then validate and run the wdl job

Set up an env variable for the log file that we can use later

In [ ]:
%env cromwell_run_log=hello_world.log

In [ ]:
!echo $cromwell_run_log

**Validate the WDL file first**

In [ ]:
!java -jar $WOMTOOL_JAR validate $wdl_filename

**Run the WDL job**

And save the output to a log file `$cromwell_run_log`

In [ ]:
!java -jar -Dconfig.file=/home/jupyter/cromwell.conf $CROMWELL_JAR run $wdl_filename -i $json_filename 2>&1 | tee $cromwell_run_log

# Now check a few things from the log file

In [ ]:
!ls *.log

Frist we can use `grep` to capture the job status 

In [ ]:
!grep "workflow finished with status" $cromwell_run_log

Or we can use a script to capture the job status and id.

In [ ]:
import re
import os

def get_cromwell_run_info(log_filename):
    """
    Parses Cromwell log for the robust 'Workflow actor' line.
    Returns: {'id': str, 'status': str}
    """
    # Handle env variable expansion if user passes '$var'
    log_filename = os.path.expandvars(log_filename)

    if not os.path.exists(log_filename):
        return {'id': None, 'status': "FILE_NOT_FOUND"}
    
    # Initialize defaults
    job_id = None
    status = "RUNNING" # Default assumption if we find an ID but no end status
    
    with open(log_filename, 'r') as f:
        for line in f:
            
            # --- STRATEGY 1: The "Gold Standard" Line ---
            # Matches: "Workflow actor for 3ab07... completed with status 'Failed'."
            # This line contains BOTH the ID and the Final Status.
            match_final = re.search(r"Workflow actor for ([a-f0-9\-]+) completed with status '(\w+)'", line)
            if match_final:
                job_id = match_final.group(1)
                status = match_final.group(2)
                # We found the definitive end of the job, so we can stop reading.
                return {'id': job_id, 'status': status}

            # --- STRATEGY 2: The "Fallback" (Start of Job) ---
            # If the job is still running, the line above doesn't exist yet.
            # So we grab the ID from the JSON blob at the start.
            if not job_id:
                match_start = re.search(r'"id":\s*"([a-f0-9\-]+)"', line)
                if match_start:
                    job_id = match_start.group(1)
                    
    return {'id': job_id, 'status': status}

In [ ]:
# 1. Use your env variable
log_filename = "$cromwell_run_log" 

# 2. Run the new function
info = get_cromwell_run_info(log_filename)
# Extract the ID into its own variable
job_id = info['id']

print(f"Job ID: {info['id']}")
print(f"Status: {info['status']}")

# 3. Simple logic to decide what to do next
if info['status'] == 'Failed':
    print("\n❌ Job failed. Searching for errors...")
    # (You can insert the error scanning script I gave you previously here)
elif info['status'] == 'Succeeded':
    print("\n✅ Job succeeded! Ready to list output files.")
else:
    print("\n⏳ Job is still running.")

In [ ]:
!echo $job_id

In [ ]:
import os 
my_bucket = os.getenv("WORKSPACE_BUCKET")
my_bucket

**And then wirte a function to capture the `$workflow_name` and `$task_name`**

In [ ]:
import re

def get_wdl_info(wdl_filename):
    """
    Parses a WDL file to extract the workflow name and the first task call.
    Returns: (workflow_name, task_folder_name)
    """
    workflow_name = None
    task_name = None

    try:
        with open(wdl_filename, 'r') as f:
            for line in f:
                # 1. Capture Workflow Name
                if not workflow_name:
                    match_wf = re.search(r'^\s*workflow\s+(\w+)', line)
                    if match_wf:
                        workflow_name = match_wf.group(1)
                
                # 2. Capture Task Call Name
                if not task_name:
                    match_task = re.search(r'^\s*call\s+(\w+)', line)
                    if match_task:
                        # Cromwell folder convention adds "call-"
                        task_name = f"call-{match_task.group(1)}"
                
                # Optimization: Stop reading if we found both
                if workflow_name and task_name:
                    break
                    
        return workflow_name, task_name

    except FileNotFoundError:
        print(f"Error: The file '{wdl_filename}' was not found.")
        return None, None

In [ ]:
!echo $wdl_filename

In [ ]:
wdl_filename = os.environ.get("wdl_filename")

In [ ]:
# Call the function and unpack the results into two variables
workflow_name, task_name = get_wdl_info(wdl_filename)

# Verify it worked
print(f"Workflow: {workflow_name}")
print(f"Task:     {task_name}")

**Wirte a function to capture the `$output_path`**

In [ ]:
import os
import re

def get_cromwell_output_path(config_file="~/cromwell.conf"):
    """
    Parses the Cromwell config file to find the Google Cloud Storage (gs://) root path.
    Returns: The gs:// path string, or None if not found.
    """
    # Expand the '~' to the full home directory path
    full_path = os.path.expanduser(config_file)
    
    try:
        with open(full_path, 'r') as f:
            for line in f:
                # Look for 'root = "gs://..."'
                # This regex ensures we ignore local paths like "/dev/null"
                match = re.search(r'root\s*=\s*"(gs://[^"]+)"', line)
                
                if match:
                    return match.group(1) # Return the path immediately
        
        print("Warning: No 'gs://' root path found in config.")
        return None

    except FileNotFoundError:
        print(f"Error: Config file not found at {full_path}")
        return None

In [ ]:
# 1. Call the function
output_path = get_cromwell_output_path()

# 2. Check and use it
if output_path:
    print(f"Ready to use: {output_path}")
    
    # You can now use it in your gsutil command combined with previous steps:
    # !gsutil ls {output_path}/{workflow_name}/{job_id}/{task_name}

In [ ]:
!echo $output_path

# Finally we can check the output files

In [ ]:
!gsutil ls {output_path}/{workflow_name}/{job_id}/{task_name}

In [ ]:
!gcloud storage cat {output_path}/{workflow_name}/{job_id}/{task_name}/stdout

# check run time

In [ ]:
import os
import re
from datetime import datetime

def calculate_process_time(log_filename):
    """
    Parses start and end timestamps from a Cromwell log and calculates duration.
    """
    # Use env var if passed as a string like "$var"
    log_filename = os.path.expandvars(log_filename)

    if not os.path.exists(log_filename):
        print(f"❌ File not found: {log_filename}")
        return

    start_time = None
    end_time = None

    # Regex to capture the timestamp: [YYYY-MM-DD HH:MM:SS,ms]
    # We look for lines starting with '[' followed by digits
    timestamp_pattern = re.compile(r"^\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{2})\]")

    with open(log_filename, 'r') as f:
        for line in f:
            match = timestamp_pattern.match(line)
            if match:
                # Get the timestamp string (e.g., "2026-01-24 02:11:17,52")
                ts_string = match.group(1)
                
                # FIX FORMAT: Python needs a dot (.) for milliseconds, not a comma (,)
                # We also assume the log is using centiseconds (2 digits), so we pad to microseconds
                clean_ts = ts_string.replace(',', '.')
                
                try:
                    current_dt = datetime.strptime(clean_ts, "%Y-%m-%d %H:%M:%S.%f")
                except ValueError:
                    # Fallback if the milliseconds parsing fails
                    clean_ts_no_ms = clean_ts.split('.')[0]
                    current_dt = datetime.strptime(clean_ts_no_ms, "%Y-%m-%d %H:%M:%S")

                # Logic:
                # 1. The FIRST timestamp we see is the Start Time
                if start_time is None:
                    start_time = current_dt
                
                # 2. We keep updating 'end_time' with every timestamp we see.
                #    The LAST one we processed by end of file will be the true End Time.
                end_time = current_dt

    # --- Calculation & Output ---
    if start_time and end_time:
        duration = end_time - start_time
        total_seconds = duration.total_seconds()
        
        # Convert to readable format (Minutes:Seconds)
        minutes = int(total_seconds // 60)
        seconds = int(total_seconds % 60)

        print(f"⏱️  PROCESS TIMING REPORT")
        print("-----------------------")
        print(f"Start:    {start_time}")
        print(f"End:      {end_time}")
        print("-----------------------")
        print(f"Duration: {duration} ({minutes}m {seconds}s)")
        print(f"Seconds:  {total_seconds:.2f} sec")
        
        return total_seconds
    else:
        print("Could not find valid timestamps in the log.")
        return 0



In [ ]:
# --- RUN IT ---
# Use your environment variable
log_file = "$cromwell_run_log"
calculate_process_time(log_file)